# ML04：PCA、K-means 与低方差信号反例

区分重建信息、聚类结构与任务相关信息；所有 fitted transforms 只读 train。

**运行方式**：上传本 `.ipynb` 到 Google Colab，连接 Python runtime，从上到下运行。
默认 CPU 即可；本 notebook 自带代码和合成数据，不需要克隆仓库、上传论文、挂载 Drive 或配置本地环境。
所有输出留空，首次运行后才产生实际结果；不要把材料准备状态当作已复现实验。

如果所选云端 runtime 缺少包，可在 Colab 新单元执行 `%pip install numpy matplotlib pandas scikit-learn torch`，
然后重启 runtime。不要为了本课程在本机安装依赖。


## 1. 环境与随机性
记录版本和 seed。换 seed 时从本单元重新运行；图中用英文标签避免依赖额外中文字体。


In [ ]:
import json
import math
import platform
import random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
SEED = 7
random.seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)
print('Python:', platform.python_version(), '| NumPy:', np.__version__, '| seed:', SEED)
import sklearn
print('sklearn:', sklearn.__version__)


## 2. PCA 和 K-means：先 train fit，再 held-out transform
只用标签评价 ARI，不把标签输入聚类。cluster ID 没有固定 class 含义。


In [ ]:
from sklearn.datasets import make_blobs
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, accuracy_score
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
X, labels = make_blobs(n_samples=700, centers=3, n_features=5, cluster_std=2., random_state=SEED)
X = X * np.array([10., 1., .2, 3., .5])
permutation = rng.permutation(len(X)); tr, ev = permutation[:500], permutation[500:]
scaler = StandardScaler().fit(X[tr])
representations = {'raw': (X[tr], X[ev]),
                   'scaled': (scaler.transform(X[tr]), scaler.transform(X[ev]))}
rows = []; fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for axis, (name, (train_x, eval_x)) in zip(axes, representations.items()):
    pca = PCA(n_components=2).fit(train_x)
    train_z, eval_z = pca.transform(train_x), pca.transform(eval_x)
    reconstruction = pca.inverse_transform(eval_z)
    cluster = KMeans(n_clusters=3, n_init=10, random_state=SEED).fit(train_z)
    predicted = cluster.predict(eval_z)
    rows.append({'representation': name, 'held_out_ARI': adjusted_rand_score(labels[ev], predicted),
                 'held_out_reconstruction_mse_in_this_scale': float(np.mean((eval_x-reconstruction)**2)),
                 'train_explained_variance_fraction': float(pca.explained_variance_ratio_.sum())})
    axis.scatter(eval_z[:, 0], eval_z[:, 1], c=predicted, s=14)
    axis.set(title=name+' PCA / K-means', xlabel='PC1', ylabel='PC2')
plt.tight_layout(); plt.show(); display(pd.DataFrame(rows))
results = {'seed': SEED, 'pca_kmeans': rows,
           'note': 'Raw/scaled reconstruction MSE use different units and are not directly comparable.'}


## 3. SVD 写出 PCA 的 projection/reconstruction


In [ ]:
train_mean = X[tr].mean(axis=0, keepdims=True)
centered = X[tr]-train_mean
U, singular_values, Vt = np.linalg.svd(centered, full_matrices=False)
components = Vt[:2].T
z = (X[ev]-train_mean) @ components
recovered = z @ components.T + train_mean
print('X:', X[ev].shape, 'components:', components.shape, 'Z:', z.shape, 'reconstruction:', recovered.shape)
plt.plot(np.cumsum(singular_values**2)/np.sum(singular_values**2), marker='o')
plt.xlabel('component index (zero-based)'); plt.ylabel('cumulative variance fraction'); plt.show()


## 4. 低方差信号也可能决定标签
两个预先固定的 pipeline 对比；不依据 held-out 结果调参。第 1 维大方差但无标签信息，第 2 维小方差却含标签。


In [ ]:
low_y = rng.integers(0, 2, size=900)
low_x = np.column_stack([rng.normal(scale=10., size=900),
                        .05*(2*low_y-1) + rng.normal(scale=.005, size=900)])
low_tr, low_ev = np.arange(650), np.arange(650, 900)
baselines = {
    'both standardized features': make_pipeline(StandardScaler(), LogisticRegression()),
    'raw PCA to one component': make_pipeline(PCA(n_components=1), StandardScaler(), LogisticRegression()),
}
counterexample = {}
for name, model in baselines.items():
    model.fit(low_x[low_tr], low_y[low_tr])
    counterexample[name] = float(accuracy_score(low_y[low_ev], model.predict(low_x[low_ev])))
results['low_variance_signal_counterexample'] = counterexample
print(counterexample)


## 面试追问与实验记录
1. PCA 最大化方差为什么不能保证保留分类信号？
2. 为什么 cluster IDs 不能直接与 labels 算 accuracy？
3. raw/scaled reconstruction MSE 能否直接比较？
4. FP2 reconstruction 很好是否必然说明接触控制有效？

记录 Question / Data / Config / Metric / Observation / Explanation / Limitation。
改变实验设置时不要反复用 test 选择答案；保存 notebook 后再下载 `.ipynb` 保留输出。


## 可选：导出指标
默认关闭。需要时打开开关；文件只写到当前 **Colab runtime** 的 `outputs/`。runtime 临时文件不等于已永久保存，可使用 Colab 文件面板下载。


In [ ]:
EXPORT_RESULTS = False
if EXPORT_RESULTS:
    destination = Path('outputs')
    destination.mkdir(exist_ok=True)
    (destination / 'metrics.json').write_text(
        json.dumps(results, indent=2, ensure_ascii=False, allow_nan=False), encoding='utf-8')
    print('Saved outputs/metrics.json in the Colab runtime; download it from the Files panel.')
